# 实验6 深度学习之RNN

## 1. RNN原理

已知拓扑结构为同步多对多RNN（基本结构见下图），输入层、隐含层（一层）、输出层的神经元均为一个，激活函数均为`ReLU`，$W = [0.4,0.5,0.2,0.15,0.3]$,$ H = [1], V = [2], S_0=0, α=0, β=0.1$。给定一个`X=[[1,1,1,1,1],[2,2,2,2,2],[3,3,3,3,3]]`的输入序列，计算其输出序列`Y`。

![RNN逻辑结构](RNN逻辑结构.png)

In [1]:
import numpy as np
X = np.array([[1, 1, 1, 1, 1], [2, 2, 2, 2, 2], [3, 3, 3, 3, 3]])
W = np.array([0.4, 0.5, 0.2, 0.15, 0.3])
H = np.array([1])
V = np.array([2])
S_0 = 0
alpha = 0
beta = 0.1

# ReLU激活函数
def ReLU(x):
    return np.maximum(x, 0)

In [2]:
# S1=f(W*X1+H*S0+beta)
S_1 = ReLU(np.dot(W, X[0]) + H * S_0 + beta)
# Y1=h(V*S1+alpha)
Y_1 = ReLU(V*S_1 + alpha)
# S2=f(W*X2+H*S1)
S_2 = ReLU(np.dot(W, X[1]) + H * S_1 + beta)
# Y2=h(V*S2+alpha)
Y_2 = ReLU(V*S_2 + alpha)
# S3=f(W*X3+H*S2)
S_3 = ReLU(np.dot(W, X[2]) + H * S_2 + beta)
# Y3=h(V*S3+alpha)
Y_3 = ReLU(V*S_3 + alpha)
print([S_1, S_2, S_3])
print([Y_1, Y_2, Y_3])

[array([1.65]), array([4.85]), array([9.6])]
[array([3.3]), array([9.7]), array([19.2])]


## 2. 基于LSTM的文本生成实践

### （1） 准备文本文件

在学在浙大中下载提供的文本文件text.txt, 也可以使用自己的文本文件，如英文小说、演讲稿等等。

### （2） 加载文本文件并进行预处理

加载并读取文本文件，然后移除换行符和回车符。注意替换文件路径。

In [3]:
data = open('text.txt').read()
data = data.replace('\n', '').replace('\r', '')

### （3） 建立字典

进行字符去重处理，为建立字典做准备，`letters`是去重后的字符列表。

In [4]:
# 字符去重处理
letters = list(set(data))
print(letters)
num_letters = len(letters)
print(num_letters)

['.', 'u', 'g', '鈥', ',', 'B', 'x', 'y', 'z', 'D', 'S', 'b', 'h', 'm', 'w', "'", 'e', 'l', 'H', 'q', 'a', 'I', 'J', 'O', 'T', 'r', 's', '攔', 'G', 'A', 'v', 'p', 'f', '攐', 't', ' ', 'i', 'W', '攁', 'c', 'd', 'n', 'k', 'o']
44


然后建立字符和整数之间的映射字典。

In [5]:
# int to char
int_to_char = {a: b for a, b in enumerate(letters)}
print(int_to_char)
# char to int
char_to_int = {b: a for a, b in enumerate(letters)}
print(char_to_int)

{0: '.', 1: 'u', 2: 'g', 3: '鈥', 4: ',', 5: 'B', 6: 'x', 7: 'y', 8: 'z', 9: 'D', 10: 'S', 11: 'b', 12: 'h', 13: 'm', 14: 'w', 15: "'", 16: 'e', 17: 'l', 18: 'H', 19: 'q', 20: 'a', 21: 'I', 22: 'J', 23: 'O', 24: 'T', 25: 'r', 26: 's', 27: '攔', 28: 'G', 29: 'A', 30: 'v', 31: 'p', 32: 'f', 33: '攐', 34: 't', 35: ' ', 36: 'i', 37: 'W', 38: '攁', 39: 'c', 40: 'd', 41: 'n', 42: 'k', 43: 'o'}
{'.': 0, 'u': 1, 'g': 2, '鈥': 3, ',': 4, 'B': 5, 'x': 6, 'y': 7, 'z': 8, 'D': 9, 'S': 10, 'b': 11, 'h': 12, 'm': 13, 'w': 14, "'": 15, 'e': 16, 'l': 17, 'H': 18, 'q': 19, 'a': 20, 'I': 21, 'J': 22, 'O': 23, 'T': 24, 'r': 25, 's': 26, '攔': 27, 'G': 28, 'A': 29, 'v': 30, 'p': 31, 'f': 32, '攐': 33, 't': 34, ' ': 35, 'i': 36, 'W': 37, '攁': 38, 'c': 39, 'd': 40, 'n': 41, 'k': 42, 'o': 43}


### （4） 字符序列预处理

下面是字符序列的预处理，它将输入的文本数据转换为适合模型训练的格式。

从输入的文本数据 data 中使用滑动窗口技术提取输入序列和目标值。例如我们使用前20个字符预测第21个字符，那么窗口大小即为20。

In [6]:
import numpy as np
from keras.utils import to_categorical

# 滑动窗口提取数据
def extract_data(data, slide):
    x = []
    y = []
    for i in range(len(data) - slide):
        x.append([a for a in data[i:i+slide]])
        y.append(data[i+slide])
    return x, y

In [7]:
# 字符到数字的批量转化
def char_to_int_Data(x, y, char_to_int):
    x_to_int = []
    y_to_int = []
    for i in range(len(x)):
        x_to_int.append([char_to_int[char] for char in x[i]])
        y_to_int.append([char_to_int[char] for char in y[i]])
    return x_to_int, y_to_int

In [8]:
# 实现输入字符文章的批量处理
def data_preprocessing(data, slide, num_letters, char_to_int):
    char_Data = extract_data(data, slide)
    int_Data = char_to_int_Data(char_Data[0], char_Data[1], char_to_int)
    Input = int_Data[0]
    Output = list(np.array(int_Data[1]). flatten())

    Input_reshaped = np.array(Input).reshape(len(Input), slide)
    new = np.random.randint(
        0, 10, size=[Input_reshaped.shape[0], Input_reshaped.shape[1], num_letters])
    for i in range(Input_reshaped.shape[0]):
        for j in range(Input_reshaped.shape[1]):
            new[i, j, :] = to_categorical(
                Input_reshaped[i, j], num_classes=num_letters)
    return new, Output

调用`data_preprocessing()` 得到可用于LSTM输入的数据。

In [9]:
# 使用前20个字符预测第21个字符
time_step = 20
X, y = data_preprocessing(data, time_step, num_letters, char_to_int)
print(X.shape)
print(len(y))

(49858, 20, 44)
49858


### （5） 划分数据集

利用`sklearn`中的`train_test_split()`划分数据集，划分比例可以是9:1。

In [10]:
# 划分数据集
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=10)
print(X_train.shape, len(y_train))

(44872, 20, 44) 44872


将`y_train`转换为 one-hot 编码格式，每个标签被转换为一个长度为`num_letters`的向量，其中对应类别的位置为1，其他位置为0。

In [11]:
from keras.utils import to_categorical
y_train_category = to_categorical(y_train, num_letters)
print(y_train_category)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


### （6） 模型建立

我们建立一个比较简单的模型，大家可以自行尝试更加复杂的模型。

**注意：可能会遇到`tensorflow`和`numpy`不兼容的情况，可以尝试降低`numpy`版本**

In [12]:
from keras.models import Sequential
from keras.layers import Dense, LSTM
model = Sequential()
model.add(LSTM(units=30, input_shape=(
    X_train.shape[1], X_train.shape[2]), activation='relu'))
model.add(Dense(units=num_letters, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 30)                9000      
                                                                 
 dense (Dense)               (None, 44)                1364      
                                                                 
Total params: 10364 (40.48 KB)
Trainable params: 10364 (40.48 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


### （7） 模型训练

In [13]:
model.fit(X_train, y_train_category, batch_size=1000, epochs=600)

Epoch 1/600


45/45 [==============================] - 2s 23ms/step - loss: 3.5413 - accuracy: 0.0998
Epoch 2/600
45/45 [==============================] - 1s 21ms/step - loss: 2.9646 - accuracy: 0.1933
Epoch 3/600
45/45 [==============================] - 1s 20ms/step - loss: 2.8571 - accuracy: 0.1933
Epoch 4/600
45/45 [==============================] - 1s 19ms/step - loss: 2.7592 - accuracy: 0.2268
Epoch 5/600
45/45 [==============================] - 1s 21ms/step - loss: 2.5603 - accuracy: 0.3045
Epoch 6/600
45/45 [==============================] - 1s 19ms/step - loss: 2.3951 - accuracy: 0.3362
Epoch 7/600
45/45 [==============================] - 1s 19ms/step - loss: 2.2915 - accuracy: 0.3620
Epoch 8/600
45/45 [==============================] - 1s 19ms/step - loss: 2.2142 - accuracy: 0.3768
Epoch 9/600
45/45 [==============================] - 1s 19ms/step - loss: 2.1480 - accuracy: 0.3937
Epoch 10/600
45/45 [==============================] - 1s 19ms/step - loss: 2.0971 - accuracy: 0.40

### （8）模型预测

使用训练好的模型对测试数据`X_test`进行预测，得到预测的类别索引。同时利用`accuracy_score`来计算模型的准确率。

In [14]:
from sklearn.metrics import accuracy_score
# y_test_predict = model.predict_classes(X_test) 这段代码在新版本的keras中已经被弃用
import numpy as np

y_test_predict = np.argmax(model.predict(X_test), axis=-1)
accuracy_test = accuracy_score(y_test, y_test_predict)
print(accuracy_test)
print(y_test_predict)
print(y_test)

156/156 [==============================] - 0s 2ms/step
0.9767348576012836
[35 12 20 ... 20 41 20]
[35, 12, 20, 43, 40, 0, 11, 14, 12, 34, 35, 17, 43, 36, 14, 1, 32, 20, 35, 20, 35, 40, 26, 26, 12, 16, 35, 34, 36, 14, 12, 1, 35, 13, 12, 1, 25, 41, 36, 17, 35, 34, 16, 39, 41, 16, 7, 26, 34, 35, 26, 16, 20, 4, 20, 40, 40, 35, 36, 35, 35, 14, 32, 16, 41, 12, 40, 35, 16, 13, 14, 20, 36, 16, 20, 17, 40, 25, 1, 20, 16, 1, 32, 11, 7, 36, 16, 2, 35, 14, 7, 20, 16, 11, 34, 20, 40, 1, 35, 34, 11, 7, 12, 21, 35, 35, 35, 34, 16, 7, 20, 41, 20, 40, 20, 41, 34, 35, 26, 41, 0, 12, 20, 21, 35, 34, 12, 36, 16, 35, 26, 39, 34, 2, 35, 20, 41, 18, 13, 17, 34, 16, 12, 41, 41, 16, 16, 26, 16, 25, 35, 17, 35, 16, 35, 31, 26, 35, 14, 41, 34, 35, 36, 36, 41, 35, 36, 40, 20, 16, 41, 35, 34, 32, 26, 34, 20, 35, 4, 20, 34, 40, 41, 35, 35, 20, 39, 16, 41, 35, 12, 20, 25, 14, 20, 16, 1, 12, 34, 21, 35, 2, 34, 12, 34, 34, 35, 20, 20, 17, 31, 12, 25, 35, 35, 35, 35, 13, 25, 12, 35, 15, 25, 16, 20, 35, 35, 13, 16, 35, 

将输出转变为字符串

In [15]:
y_test_predict_char = [int_to_char[a] for a in y_test_predict]
y_test_char = [int_to_char[a] for a in y_test]
print(y_test_predict_char)
print(y_test_char)

[' ', 'h', 'a', 'o', 'd', '.', 'b', 'w', 'h', 't', ' ', 'l', 'o', 'i', 'a', 'u', 'f', 'a', ' ', 'a', ' ', ' ', 's', 's', 'h', 'e', ' ', 't', 'i', 'w', 'h', 'u', ' ', 'm', 'h', 'u', 'r', 'n', 'i', 'l', ' ', 't', 'e', 'c', 'n', 'e', 'y', 's', 't', ' ', 's', 'e', 'a', ',', 'a', 'd', 'd', ' ', 'i', ' ', ' ', 'w', 'f', 'e', 'n', 'h', 'd', ' ', 'e', 'm', 'w', 'a', 'i', 'e', 'a', 'l', 'd', 'r', 'u', 'a', 'e', 'u', 'f', 'b', 'y', 'i', 'e', 'g', ' ', 'w', 'y', 'a', 'e', 'b', 't', 'a', 'd', 'u', ' ', 'e', 'b', 'y', 'h', 'I', ' ', ' ', ' ', 't', 'e', 'y', 'a', 'n', 'a', 'd', 'a', 'n', 't', ' ', 's', 'n', '.', 'h', 'a', 'I', ' ', 't', 'h', 'i', 'e', ' ', 's', 'c', 't', 'g', ' ', 'a', 'n', 'H', 'm', 'l', 't', 'e', 'h', 'n', 'n', 'e', 'e', 's', 'e', 'r', ' ', 'l', ' ', 'e', ' ', 'p', 's', ' ', 'a', 'n', 't', ' ', 'i', 'i', 'n', ' ', 'i', 'd', 'a', 'e', 'n', ' ', 't', 'f', 's', 'e', 'a', ' ', ',', 'a', 't', 'd', 'n', ' ', ' ', 'a', 'c', 'e', 'n', ' ', 'h', 'a', 'r', 'w', 'a', 'e', 'u', 'h', 't', 'I',

对于文本生成任务，类别索引代表模型预测的字符或单词，但这些索引对我们来说并不直观。因此，通常会将预测结果的类别索引转化回实际的文本字符，以便理解。

我们可以选取文本文件中存在的一小节文本，然后预测后续的文本。首先得到索引序列。

In [16]:
new_letters = 'He had built his wealth on the strength of his determination'
# -------------------------实现预测代码-------------------------
# 数据处理
X, y = data_preprocessing(new_letters, 20, num_letters, char_to_int)
print(X.shape)
# 进行预测
y_new_predict = model.predict(X)
y_new_predict = np.argmax(y_new_predict, axis=-1)

# --------------------------------------------------------------
print(y_new_predict)

(40, 20, 44)
2/2 [==============================] - 0s 3ms/step
[17 34 12 35 43 41 35 34 12 16 35 26 34 25 16 41  2 34 12 35 43 32 35 12
 36 26 35 40 16 16 16 25 13 36 41 20 34 36 43 41]


然后转化为字符。

In [17]:
y_new_predict_char = [int_to_char[i] for i in y_new_predict]
print(''.join(y_new_predict_char))

lth on the strength of his deeermination
